In [6]:
CENSUS_API_KEY = "1ea2fcb7a848498db50b081676b2483089a2852a"
BEA_API_KEY = "CEDA997C-4E9E-49D8-B931-4EBBF9091AFC"

In [7]:
"""
Skrypt do zbierania danych dla wszystkich stanów USA z publicznych API.
Cel: zmienne wyjaśniające odsetek ludzi uprawiających sport w stanach USA.

Źródła:
1. Census ACS 5-Year  – dochód, demografia, wykształcenie, ubóstwo
2. BLS LAUS           – stopa bezrobocia
3. CDC PLACES         – zachowania zdrowotne (binge drinking, otyłość, LPA, ...)
4. CDC BRFSS          – aktywność fizyczna, otyłość, alkohol
5. Census CBP         – liczba siłowni i sklepów alkoholowych
6. EPA AQS            – jakość powietrza (PM2.5)
7. County Health Rankings (RWJF) – dostęp do parków, przestępczość
8. CDC NPAO           – aktywność fizyczna i otyłość (bezpośrednie dane BRFSS)

Wymagania:
    pip install requests pandas tqdm openpyxl

Klucze API (bezpłatne):
    Census: https://api.census.gov/data/key_signup.html
    Ustaw: export CENSUS_API_KEY='twój_klucz'
    Bez klucza ACS/CBP też działa, ale z limitem zapytań.
"""

import os, time, pickle, warnings
import requests
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ── Klucze API ─────────────────────────────────────────────────────────────────
# CENSUS_API_KEY = os.getenv("CENSUS_API_KEY", "")
# BLS_API_KEY    = os.getenv("BLS_API_KEY", "")

STATES = {
    "01":"AL","02":"AK","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT",
    "10":"DE","11":"DC","12":"FL","13":"GA","15":"HI","16":"ID","17":"IL",
    "18":"IN","19":"IA","20":"KS","21":"KY","22":"LA","23":"ME","24":"MD",
    "25":"MA","26":"MI","27":"MN","28":"MS","29":"MO","30":"MT","31":"NE",
    "32":"NV","33":"NH","34":"NJ","35":"NM","36":"NY","37":"NC","38":"ND",
    "39":"OH","40":"OK","41":"OR","42":"PA","44":"RI","45":"SC","46":"SD",
    "47":"TN","48":"TX","49":"UT","50":"VT","51":"VA","53":"WA","54":"WV",
    "55":"WI","56":"WY",
}


# ── Pomocnicze funkcje ─────────────────────────────────────────────────────────
def safe_get(url, params=None, label="", timeout=30):
    """GET z pełną obsługą błędów. Zwraca response lub None."""
    try:
        resp = requests.get(url, params=params, timeout=timeout)
    except requests.RequestException as e:
        print(f"  [{label}] Błąd połączenia: {e}")
        return None
    if resp.status_code != 200:
        print(f"  [{label}] HTTP {resp.status_code}: {resp.text[:250]}")
        return None
    if not resp.text or resp.text.strip() in ("", "[]", "null"):
        print(f"  [{label}] Pusta odpowiedź")
        return None
    return resp


def safe_json(resp, label=""):
    """Parsuje JSON; zwraca None przy błędzie."""
    try:
        return resp.json()
    except Exception as e:
        print(f"  [{label}] JSON error: {e} | tekst: {resp.text[:250]}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# 1. CENSUS ACS 5-Year (2011–2023)
# ══════════════════════════════════════════════════════════════════════════════
ACS_VARS = {
    "B19301_001E": "per_capita_income",
    "B01003_001E": "total_population",
    "B19013_001E": "median_household_income",
    "B17001_002E": "population_below_poverty",
    "B15003_022E": "edu_bachelors_25plus",
    "B15003_023E": "edu_masters_25plus",
    "B15003_025E": "edu_doctoral_25plus",
    "B23025_005E": "civilian_unemployed",
    "B23025_002E": "labor_force",
    "B01002_001E": "median_age",
    "B02001_002E": "pop_white_alone",
    "B02001_003E": "pop_black_alone",
    "B02001_005E": "pop_asian_alone",
    "B03001_003E": "pop_hispanic",
    "B25077_001E": "median_home_value",
    "B08301_010E": "commute_public_transit",
    "B08303_001E": "commute_total",
}


def fetch_acs(year):
    variables = ",".join(ACS_VARS.keys())
    key_part = f"&key={CENSUS_API_KEY}" if CENSUS_API_KEY else ""
    url = (
        f"https://api.census.gov/data/{year}/acs/acs5"
        f"?get=NAME,{variables}&for=state:*{key_part}"
    )
    resp = safe_get(url, label=f"ACS {year}")
    if resp is None:
        return pd.DataFrame()
    data = safe_json(resp, label=f"ACS {year}")
    if not data or len(data) < 2:
        return pd.DataFrame()
    df = pd.DataFrame(data[1:], columns=data[0])
    df = df.rename(columns=ACS_VARS)
    df["year"] = year
    df["state_fips"] = df["state"]
    df["state_abbr"] = df["state_fips"].map(STATES)
    num_cols = list(ACS_VARS.values())
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
    return df[["year", "state_fips", "state_abbr", "NAME"] + num_cols]


def collect_acs():
    print("\n=== 1. Census ACS 5-Year (dochód, demografia) 2011-2023 ===")
    frames = []
    for yr in tqdm(range(2011, 2024), desc="ACS"):
        df = fetch_acs(yr)
        if not df.empty:
            frames.append(df)
        time.sleep(0.4)
    result = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print(f"  → {result.shape}")
    return result


# ══════════════════════════════════════════════════════════════════════════════
# 2. BLS LAUS – stopa bezrobocia (2011–2023)
# ══════════════════════════════════════════════════════════════════════════════
def collect_bls():
    print("\n=== 2. BLS LAUS – stopa bezrobocia 2011-2023 ===")
    series_map = {f"LASST{fips}0000000000003": fips for fips in STATES}
    all_ids = list(series_map.keys())
    chunk_size = 25 if not BLS_API_KEY else 50
    frames = []
    url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

    for i in tqdm(range(0, len(all_ids), chunk_size), desc="BLS chunks"):
        chunk = all_ids[i: i + chunk_size]
        payload = {"seriesid": chunk, "startyear": "2011", "endyear": "2023", "annualaverage": True}
        if BLS_API_KEY:
            payload["registrationkey"] = BLS_API_KEY
        try:
            resp = requests.post(url, json=payload, timeout=60)
            if resp.status_code != 200:
                print(f"  [BLS chunk {i}] HTTP {resp.status_code}")
                time.sleep(2)
                continue
            result = resp.json()
        except Exception as e:
            print(f"  [BLS chunk {i}] Błąd: {e}")
            time.sleep(2)
            continue

        for series in result.get("Results", {}).get("series", []):
            sid = series["seriesID"]
            fips = series_map.get(sid, "??")
            for obs in series.get("data", []):
                if obs.get("period") == "M13":
                    val = obs.get("value", "-")
                    frames.append({
                        "year": int(obs["year"]),
                        "state_fips": fips,
                        "state_abbr": STATES.get(fips, "??"),
                        "unemployment_rate_bls": float(val) if val != "-" else None,
                    })
        time.sleep(1.2)

    result = pd.DataFrame(frames) if frames else pd.DataFrame()
    print(f"  → {result.shape}")
    return result


# ══════════════════════════════════════════════════════════════════════════════
# 3. CDC PLACES – zachowania zdrowotne (2016–2023)
# ══════════════════════════════════════════════════════════════════════════════
PLACES_MEASURES = [
    "BINGE","CSMOKING","LPA","OBESITY","SLEEP","DEPRESSION",
    "DIABETES","HIGHCHOL","HYPERTEN","CHECKUP","DENTAL",
    "ARTHRITIS","CASTHMA","CHD","COPD","KIDNEY","STROKE",
]


def fetch_places():
    print("\n=== 3. CDC PLACES – zachowania zdrowotne (poziom stanu) ===")
    url = "https://data.cdc.gov/resource/dv4u-3x3q.json"
    measures_filter = ",".join(f"'{m}'" for m in PLACES_MEASURES)
    all_rows, limit, offset = [], 5000, 0

    while True:
        params = {
            "$where": f"measureid in({measures_filter})",
            "$limit": limit,
            "$offset": offset,
            "$select": "year,locationabbr,locationdesc,measureid,data_value,data_value_type,low_confidence_limit,high_confidence_limit",
        }
        resp = safe_get(url, params=params, label=f"PLACES offset={offset}", timeout=60)
        if resp is None:
            break
        batch = safe_json(resp, label=f"PLACES offset={offset}")
        if not batch:
            break
        all_rows.extend(batch)
        if len(batch) < limit:
            break
        offset += limit
        time.sleep(0.5)

    if not all_rows:
        print("  → Brak danych PLACES")
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    df = df.rename(columns={
        "locationabbr": "state_abbr", "locationdesc": "state_name",
        "data_value": "value", "data_value_type": "value_type",
        "low_confidence_limit": "ci_low", "high_confidence_limit": "ci_high",
    })
    for c in ["value", "ci_low", "ci_high"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    crude = df[df["value_type"] == "Crude prevalence"].copy()
    if crude.empty:
        crude = df.copy()
    pivot = crude.pivot_table(
        index=["year", "state_abbr", "state_name"],
        columns="measureid", values="value", aggfunc="first",
    ).reset_index()
    pivot.columns.name = None
    pivot.columns = [
        f"places_{c.lower()}_pct" if c not in ("year","state_abbr","state_name") else c
        for c in pivot.columns
    ]
    print(f"  → {pivot.shape}")
    return pivot


# ══════════════════════════════════════════════════════════════════════════════
# 4. CDC BRFSS Prevalence & Trends (2011–2023)
# ══════════════════════════════════════════════════════════════════════════════
def fetch_brfss():
    print("\n=== 4. CDC BRFSS – aktywność fizyczna + inne (2011-2023) ===")
    url = "https://data.cdc.gov/resource/dttw-5yxu.json"
    all_rows, limit, offset = [], 10000, 0
    topics = [
        "Exercise", "Overweight", "Alcohol Consumption",
        "Fruits and Vegetables", "Arthritis", "Asthma",
        "Diabetes", "Heart Disease", "Hypertension", "Mental Health",
    ]
    topics_filter = ",".join(f"'{t}'" for t in topics)

    while True:
        params = {
            "$where": f"topic in({topics_filter})",
            "$limit": limit,
            "$offset": offset,
            "$select": (
                "year,locationabbr,locationdesc,topic,response,"
                "data_value,data_value_type,sample_size,gender,age_group,race_ethnicity"
            ),
        }
        resp = safe_get(url, params=params, label=f"BRFSS offset={offset}", timeout=90)
        if resp is None:
            break
        batch = safe_json(resp, label=f"BRFSS offset={offset}")
        if not batch:
            break
        all_rows.extend(batch)
        if len(batch) < limit:
            break
        offset += limit
        time.sleep(0.6)

    if not all_rows:
        print("  → Brak danych BRFSS")
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    df["data_value"] = pd.to_numeric(df["data_value"], errors="coerce")
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df = df.rename(columns={"locationabbr": "state_abbr", "locationdesc": "state_name"})
    print(f"  → {df.shape} (surowe – nie-pivot)")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# 5. Census CBP – siłownie (713940) + sklepy alkoholowe (445310)
# ══════════════════════════════════════════════════════════════════════════════
CBP_NAICS = {"713940": "fitness_centers", "445310": "liquor_stores"}


def fetch_cbp_year(year, naics):
    naics_var = "NAICS2017" if year >= 2017 else "NAICS2012"
    key_part = f"&key={CENSUS_API_KEY}" if CENSUS_API_KEY else ""
    url = (
        f"https://api.census.gov/data/{year}/cbp"
        f"?get={naics_var},ESTAB,EMP,PAYANN"
        f"&for=state:*&{naics_var}={naics}{key_part}"
    )
    resp = safe_get(url, label=f"CBP {year} {naics}")
    if resp is None:
        return pd.DataFrame()
    data = safe_json(resp, label=f"CBP {year} {naics}")
    if not data or len(data) < 2:
        return pd.DataFrame()
    df = pd.DataFrame(data[1:], columns=data[0])
    df["year"] = year
    df["category"] = CBP_NAICS[naics]
    df["state_fips"] = df["state"]
    df["state_abbr"] = df["state_fips"].map(STATES)
    for c in ["ESTAB", "EMP", "PAYANN"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    cols = ["year", "state_fips", "state_abbr", "category"] + [c for c in ["ESTAB","EMP","PAYANN"] if c in df.columns]
    return df[cols]


def collect_cbp():
    print("\n=== 5. Census CBP – siłownie i sklepy alkoholowe (2011-2023) ===")
    frames = []
    for yr in tqdm(range(2011, 2024), desc="CBP"):
        for naics in CBP_NAICS:
            df = fetch_cbp_year(yr, naics)
            if not df.empty:
                frames.append(df)
            time.sleep(0.35)

    if not frames:
        return pd.DataFrame()
    combined = pd.concat(frames, ignore_index=True)
    pivot = combined.pivot_table(
        index=["year", "state_fips", "state_abbr"],
        columns="category", values="ESTAB", aggfunc="sum",
    ).reset_index()
    pivot.columns.name = None
    pivot.columns = [
        f"{c}_estab" if c not in ("year","state_fips","state_abbr") else c
        for c in pivot.columns
    ]
    print(f"  → {pivot.shape}")
    return pivot


# ══════════════════════════════════════════════════════════════════════════════
# 6. EPA AQS – PM2.5 (2011–2023)
# ══════════════════════════════════════════════════════════════════════════════
def collect_epa_pm25():
    print("\n=== 6. EPA AQS – PM2.5 (2011-2023) ===")
    EPA_EMAIL = os.getenv("EPA_AQS_EMAIL", "test@aqs.api")
    EPA_KEY   = os.getenv("EPA_AQS_KEY",   "test")
    frames = []

    for year in tqdm(range(2011, 2024), desc="EPA PM2.5"):
        url = "https://aqs.epa.gov/data/api/annualData/byState"
        params = {
            "email": EPA_EMAIL, "key": EPA_KEY,
            "param": "88101",
            "bdate": f"{year}0101", "edate": f"{year}1231",
            "state": "ALL",
        }
        resp = safe_get(url, params=params, label=f"EPA AQS {year}", timeout=60)
        if resp is None:
            time.sleep(1)
            continue
        data = safe_json(resp, label=f"EPA AQS {year}")
        if not data:
            time.sleep(1)
            continue
        rows = data.get("Data", [])
        if not rows:
            time.sleep(1)
            continue
        for row in rows:
            frames.append({
                "year": year,
                "state_fips": row.get("state_code"),
                "state_abbr": STATES.get(row.get("state_code",""), row.get("state_code","")),
                "pm25_annual_mean": row.get("arithmetic_mean"),
                "pm25_obs_count":   row.get("observation_count"),
            })
        time.sleep(0.8)

    if not frames:
        print("  → Brak danych PM2.5 (zarejestruj się na aqs.epa.gov dla pełnych danych)")
        return pd.DataFrame()

    df = pd.DataFrame(frames)
    df["pm25_annual_mean"] = pd.to_numeric(df["pm25_annual_mean"], errors="coerce")
    agg = (df.groupby(["year","state_fips","state_abbr"])
             .agg(pm25_mean=("pm25_annual_mean","mean"),
                  pm25_stations=("pm25_annual_mean","count"))
             .reset_index())
    print(f"  → {agg.shape}")
    return agg


# ══════════════════════════════════════════════════════════════════════════════
# 7. County Health Rankings (RWJF) – dostęp do parków, przestępczość
# ══════════════════════════════════════════════════════════════════════════════
def fetch_chr():
    print("\n=== 7. County Health Rankings – pliki Excel (2016-2023) ===")
    chr_urls = {
        2023: "https://www.countyhealthrankings.org/sites/default/files/media/document/2023%20County%20Health%20Rankings%20State%20Data%20-%20v1.xlsx",
        2022: "https://www.countyhealthrankings.org/sites/default/files/media/document/2022%20County%20Health%20Rankings%20State%20Data%20-%20v1.xlsx",
        2021: "https://www.countyhealthrankings.org/sites/default/files/media/document/2021%20County%20Health%20Rankings%20State%20Data%20-%20v1_0.xlsx",
        2020: "https://www.countyhealthrankings.org/sites/default/files/media/document/2020%20County%20Health%20Rankings%20State%20Data%20-%20v2.xlsx",
        2019: "https://www.countyhealthrankings.org/sites/default/files/media/document/2019%20County%20Health%20Rankings%20State%20Data%20-%20v1_0.xlsx",
        2018: "https://www.countyhealthrankings.org/sites/default/files/media/document/2018%20County%20Health%20Rankings%20State%20Data%20-%20v3.xlsx",
        2017: "https://www.countyhealthrankings.org/sites/default/files/media/document/2017%20County%20Health%20Rankings%20State%20Data%20-%20v2.xlsx",
        2016: "https://www.countyhealthrankings.org/sites/default/files/media/document/2016%20County%20Health%20Rankings%20State%20Data%20-%20v3.xlsx",
    }

    frames = []
    for year, url in tqdm(chr_urls.items(), desc="CHR Excel"):
        try:
            resp = requests.get(url, timeout=90)
            if resp.status_code != 200:
                print(f"  [CHR {year}] HTTP {resp.status_code}")
                continue
            from io import BytesIO
            xls = pd.ExcelFile(BytesIO(resp.content))
            target = None
            for sheet in xls.sheet_names:
                if any(kw in sheet.lower() for kw in ["ranked measure","additional measure","outcomes"]):
                    target = sheet
                    break
            if target is None:
                target = xls.sheet_names[0]
            df = pd.read_excel(xls, sheet_name=target, header=1)
            df["year_chr"] = year
            frames.append(df)
            print(f"  [CHR {year}] OK – arkusz '{target}', {df.shape}")
        except Exception as e:
            print(f"  [CHR {year}] Błąd: {e}")
        time.sleep(1.5)

    if not frames:
        return pd.DataFrame()
    result = pd.concat(frames, ignore_index=True)
    print(f"  → {result.shape} (surowe CHR)")
    return result


# ══════════════════════════════════════════════════════════════════════════════
# 8. CDC NPAO – aktywność fizyczna i otyłość (2011–2023)
#    Najlepsza zmienna zależna: % dorosłych bez aktywności fizycznej w czasie wolnym
# ══════════════════════════════════════════════════════════════════════════════
def fetch_npao():
    print("\n=== 8. CDC NPAO – aktywność fizyczna i otyłość (2011-2023) ===")
    url = "https://data.cdc.gov/resource/4br2-t265.json"
    all_rows, limit, offset = [], 10000, 0

    while True:
        params = {
            "$limit": limit,
            "$offset": offset,
            "$select": (
                "yearstart,yearend,locationabbr,locationdesc,"
                "class,topic,question,data_value,data_value_type,"
                "data_value_unit,low_confidence_limit,high_confidence_limit,"
                "sample_size,stratificationcategory1,stratification1"
            ),
        }
        resp = safe_get(url, params=params, label=f"NPAO offset={offset}", timeout=90)
        if resp is None:
            break
        batch = safe_json(resp, label=f"NPAO offset={offset}")
        if not batch:
            break
        all_rows.extend(batch)
        if len(batch) < limit:
            break
        offset += limit
        time.sleep(0.5)

    if not all_rows:
        print("  → Brak danych NPAO")
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    df = df.rename(columns={
        "yearstart": "year_start", "yearend": "year_end",
        "locationabbr": "state_abbr", "locationdesc": "state_name",
        "data_value": "value", "data_value_type": "value_type",
        "low_confidence_limit": "ci_low", "high_confidence_limit": "ci_high",
        "stratificationcategory1": "strat_category", "stratification1": "stratification",
    })
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["year_start"] = pd.to_numeric(df["year_start"], errors="coerce")
    print(f"  → {df.shape} (surowe – zawiera pytania o aktywność fizyczną)")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
def main():
    print("=" * 65)
    print("  Zbieranie danych dla stanów USA – zmienne wyjaśniające sport")
    print("=" * 65)

    if not CENSUS_API_KEY:
        print("\n⚠  Brak klucza Census API.")
        print("   ACS i CBP działają bez klucza, ale wolniej.")
        print("   Rejestracja: https://api.census.gov/data/key_signup.html\n")

    results = {}
    steps = [
        ("acs",                    collect_acs),
        ("bls_unemployment",       collect_bls),
        ("cdc_places",             fetch_places),
        ("cdc_brfss",              fetch_brfss),
        ("cbp_businesses",         collect_cbp),
        ("epa_pm25",               collect_epa_pm25),
        ("county_health_rankings", fetch_chr),
        ("cdc_npao",               fetch_npao),
    ]

    for name, func in steps:
        try:
            df = func()
            results[name] = df
        except Exception as e:
            print(f"\n  !! Nieoczekiwany błąd w {name}: {e}")
            results[name] = pd.DataFrame()

    print("\n" + "=" * 65)
    print("  Zapisywanie")
    print("=" * 65)
    saved = []
    for name, df in results.items():
        if df is not None and not df.empty:
            path = f"{name}.csv"
            df.to_csv(path, index=False)
            print(f"  ✓  {path:45s} {str(df.shape):>15}")
            saved.append(path)
        else:
            print(f"  –  {name:45s} {'brak danych':>15}")

    with open("all_data.pkl", "wb") as f:
        pickle.dump(results, f)

    print(f"\nZapisano {len(saved)} CSV + all_data.pkl")
    print("\nJak załadować:")
    print("  import pickle")
    print("  with open('all_data.pkl','rb') as f: data = pickle.load(f)")
    print("  # Kluczowe DataFrames:")
    print("  df_npao  = data['cdc_npao']    # aktywność fizyczna (zmienna zależna)")
    print("  df_brfss = data['cdc_brfss']   # exercise, otyłość, alkohol")
    print("  df_acs   = data['acs']         # dochód, demografia")
    print("  df_bls   = data['bls_unemployment']  # bezrobocie")
    return results


if __name__ == "__main__":
    results = main()

  Zbieranie danych dla stanów USA – zmienne wyjaśniające sport

=== 1. Census ACS 5-Year (dochód, demografia) 2011-2023 ===


ACS:   0%|          | 0/13 [00:00<?, ?it/s]

  [ACS 2011] HTTP 400: error: unknown variable 'B15003_022E'


ACS: 100%|██████████| 13/13 [00:21<00:00,  1.65s/it]


  → (624, 21)

=== 2. BLS LAUS – stopa bezrobocia 2011-2023 ===


...

CHR Excel:  88%|████████▊ | 7/8 [00:07<00:01,  1.11s/it]

  [CHR 2017] HTTP 404


CHR Excel: 100%|██████████| 8/8 [00:08<00:00,  1.12s/it]

  [CHR 2016] HTTP 404

=== 8. CDC NPAO – aktywność fizyczna i otyłość (2011-2023) ===


  [NPAO offset=0] HTTP 404: {
  "code" : "dataset.missing",
  "error" : true,
  "message" : "Not found",
  "data" : {
    "id" : "4br2-t265"
  }
}

  → Brak danych NPAO

  Zapisywanie
  ✓  acs.csv                                             (624, 21)
  –  bls_unemployment                                  brak danych
  –  cdc_places                                        brak danych
  –  cdc_brfss                                         brak danych
  ✓  cbp_businesses.csv                                   (612, 5)
  –  epa_pm25                                          brak danych
  –  county_health_rankings                            brak danych
  –  cdc_npao                                          brak danych

Zapisano 2 CSV + all_data.pkl

Jak załadować:
  import pickle
  with open('all_data.pkl','rb') as f: data = pickle.load(f)
  # Kluczowe DataFrames:
  df_npao  = data['cdc_npao']    # aktywność fizyczna (zmienna zależna)
  df_brfss = data['cdc_brfss']   # exercise, otyłość, alkoh